# Ayiru × LangChain — agent savings demo

**What this shows.** A LangChain `AyiruTool` answering 10 developer-tool questions against a local Ayiru backend. We don't drive an LLM — we invoke the tool directly so the result is deterministic and runnable without API keys. Each cell prints HIT/MISS, then the last cell pulls `/v1/stats/savings` and shows the running token-savings tally.

**Preconditions.**
1. Ayiru backend running on `http://localhost:8000` (the demo expects the post-Stage-20 bulk graph: ~40 tools / ~82 claims).
2. `pip install 'ayiru-client[langchain]'` in the kernel.

If you don't have the bulk graph yet, the curated 5 tools (docker, git, github-cli, vercel-cli, openai-api) are enough to see hits — the bulk graph just widens the hit list past the curated set.

**What you should see at the bottom.** A footer like *"saved ~4,800 tokens ≈ $0.014 across this run."* That number is the v0.2 product thesis materialized: every HIT means the agent didn't have to round-trip through `web_search`.

In [ ]:
import json

from ayiru_client import Ayiru
from ayiru_client.langchain import AyiruTool

BASE_URL = "http://localhost:8000"

client = Ayiru(base_url=BASE_URL)
tool = AyiruTool(client=client)

print(f"Tool ready — name={tool.name!r}")
print(f"Description (first 200 chars): {tool.description[:200]}…")

## 10-question batch

Seven we expect to hit the v0.2 graph (curated tools + bulk-ingested CLIs), three we expect to miss (out of scope — the agent should fall back to web_search).

In [ ]:
QUESTIONS = [
    # --- expected HITs (curated + bulk-ingested tools) ---
    "how do I list docker volumes",
    "how do I delete a github repo with gh",
    "what does git log do",
    "how do I authenticate with the openai api",
    "what does kubectl describe pod do",
    "how do I install a helm chart",
    "how do I install a package with apt",
    # --- expected MISSes (out of scope for the v0.2 graph) ---
    "how do I configure my ergonomic keyboard",
    "what is the best programming language for embedded systems",
    "what is the airspeed velocity of an unladen swallow",
]

hits, misses = 0, 0
tokens_saved_local = 0

for i, q in enumerate(QUESTIONS, start=1):
    raw = tool.invoke({"question": q})
    payload = json.loads(raw)
    if payload["fallback_recommended"]:
        misses += 1
        print(f"  [{i:>2}] MISS  →  {q}")
    else:
        hits += 1
        top = payload["answers"][0]
        tokens_saved_local += payload["estimated_tokens_saved"]
        print(f"  [{i:>2}] HIT   →  {q}")
        print(f"           tool={top['tool_id']}  confidence={top['confidence']:.2f}  level={top['verification_level']}")
        print(f"           {top['statement'][:140]}")

print(f"\nLocal tally: {hits} HITs / {misses} MISSes  (target: 7 / 3)")
print(f"Tokens saved this run (per-response sum): {tokens_saved_local:,}")

## Server-side aggregate (the moat-as-data)

Every HIT above emitted a `QUERY_SERVED` audit event server-side. The `savings()` endpoint replays those events over a time window and reports the aggregate token savings + estimated USD value. This is the footer agent dev teams can put in a dashboard to justify keeping Ayiru in the stack.

In [ ]:
savings = client.savings("24h")

print(f"window:                   {savings.window}")
print(f"total queries served:     {savings.total_queries_served:,}")
print(f"  of those, fallbacks:    {savings.fallback_count:,}")
print(f"  hit rate:               {(1 - savings.fallback_count / max(savings.total_queries_served, 1)):.0%}")
print(f"total tokens saved:       {savings.total_tokens_saved:,}")
print(f"estimated USD saved:      ${savings.estimated_usd_saved:.4f}")
print(f"  (priced at ${savings.usd_per_million_input_tokens}/M input tokens)")

print()
print(f"→ saved ~{savings.total_tokens_saved:,} tokens ≈ ${savings.estimated_usd_saved:.4f} across the last {savings.window}.")

client.close()

## Next steps

To wire this into a real LangChain agent loop, replace the direct `tool.invoke(...)` calls with an agent that has `[AyiruTool(...), TavilySearch(...)]` (or your favorite web-search tool) in its tool list. The `description` field on `AyiruTool` is tuned to make the agent prefer Ayiru over web search for stable technical questions — that's the whole point.

See the [Ayiru docs](https://github.com/ruth411/ayiru) for the architecture, the verification ladder (L0–L5), and how to submit your own curated claims.